# Solar EoL Transportation Cost Estimator

This project focuses on estimating the cost of transporting EoL solar PV modules from all power stations in Australia to nearest supplier/buyer of each recoverable material. 

This project also show the difference in estimated transportation cost of Conventional and Mobile recycling method.

Furthermore, this project also include the estimated monetary value of recoverable materials in order to show how many modules should be recycled in one go so we reach a break-even point or obtain profit.

#### Assumptions

To simplify the calculation, we decided to use the following assumptions.
1. "Photovoltaic solar panels consist of 95% recyclable materials" (Clean Energy Council, 2025). This means almost 100% of PV components can be recycled and sold. Hence, our assumption is 100% of PV materials can be recycled and would be available to be sold to supplier.
2. The composition of each material in each PV module is as follows.
3. For estimating monetary value and converting different units, we use the following estimation. \
    30kg/panel \
    0.6kW/panel \
    50kg/kW \
`10$/kw = worth of panel power` \
`10$/50kg = 0.2$/kg = worth of panel waste`
4. The capacity of solar EoL panel is 300 W per module.
5. Truck capacity for transporting EoL modules: 8,000 kg.

#### Install necessary packages

In [1]:
!pip install beautifulsoup4 pandas folium

In [8]:
!pip install googlemaps

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40749 sha256=9b567f26d5118af86f7c1bed035e8e9a893fedd8f113a5c6f1fc34afe203c7e1
  Stored in directory: c:\users\lenovo\appdata\local\pip\cache\wheels\76\2a\24\5993a7b77c9a37b86f415096436a448c1babdd132066bdcb31
Successfully built googlemaps


  DEPRECATION: Building 'googlemaps' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'googlemaps'. Discussion can be found at https://github.com/pypa/pip/issues/6334


#### Extract position of all solar panel power stations

In [3]:
import pandas as pd
from bs4 import BeautifulSoup
import re

# 1. Load the HTML content from your saved .txt file
with open('power_stations.txt', 'r', encoding='utf-8') as file:
    soup = BeautifulSoup(file.read(), 'html.parser')

# 2. Find the table containing the data
table = soup.find('table', class_='power-stations-data-table')
data = []

# 3. Loop through every row in the table body
for row in table.find('tbody').find_all('tr'):
    cols = row.find_all('td')
    
    # Ensure the row has the correct number of columns
    if len(cols) >= 8:
        name = cols[0].text.strip()
        size = cols[3].text.strip()
        
        # Extract the raw link exactly as it is written in your HTML
        link_tag = cols[7].find('a')
        link = link_tag['href'] if link_tag and 'href' in link_tag.attrs else None
        
        lat, lon = None, None
        
        # 4. Use Regex to extract the coordinates directly from the link string
        if link:
            # This looks for the '@' symbol, grabs the negative/positive decimal, 
            # skips the comma, and grabs the second decimal.
            coords_match = re.search(r'@([-]?\d+\.\d+),([-]?\d+\.\d+)', link)
            
            if coords_match:
                lat = float(coords_match.group(1))
                lon = float(coords_match.group(2))
        
        # Append the extracted info to our list
        data.append({
            'Name': name,
            'Size (kW)': size,
            'Link': link,
            'Latitude': lat,
            'Longitude': lon
        })

In [4]:
# Convert to a Pandas DataFrame
df = pd.DataFrame(data)
df

,Name,Size (kW),Link,Latitude,Longitude
0,100 Harris Street Pyrmont,199.9,"https://www.google.com/maps/@-33.86835036,151....",-33.868350,151.193742
1,110 Somersby Falls Rd,199.1,"https://www.google.com/maps/@-33.410577,151.27...",-33.410577,151.279734
2,115 Frederick Street,292.0,"https://www.google.com/maps/@-27.387098,153.07...",-27.387098,153.075908
3,1182 Old Port Rd Power Station,199.7,"https://www.google.com/maps/@-34.86368301,138....",-34.863683,138.508196
4,123 Sippy Downs,199.6,"https://www.google.com/maps/@-26.713876,153.06...",-26.713876,153.061123
...,...,...,...,...,...
2873,Zammit Ham 2,281.1,"https://www.google.com/maps/@-33.798161,150.95...",-33.798161,150.956296
2874,ZENEXUS,250.3,"https://www.google.com/maps/@-27.58794,152.878...",-27.587940,152.878980
2875,Zerella Virginia,997.0,"https://www.google.com/maps/@-34.6397,138.5889...",-34.639700,138.588900
2876,ZEUSAPPOLLO SOLAR TC Do,141.6,"https://www.google.com/maps/@-31.3514,115.5661...",-31.351400,115.566100


#### List of Recycling Hub

#### List of Supplier/Buyer

## Conventional Recycling

### Number of Truck Needed

In [11]:
import math

# Choosing the unit user want to input
# Ask user to input
# Convert into 2 other units, and show these converted number


# Ask for user input in kW
kw_input = float(input("Enter the amount of solar panels to recycle (in kW): "))

# Calculate the average number of panels (0.6 kW = 1 panel)
panels_exact = kw_input / 0.6
total_panels = math.ceil(panels_exact)

# Convert kW to kg (1 kW = 50 kg)
total_kg = kw_input * 50

# Calculate trucks needed based on weight limit (29,000 kg per truck)
trucks_exact = total_kg / 29000
trucks_needed = math.ceil(trucks_exact)

# Display the final breakdown
print("\n--- Logistics Breakdown ---")
print(f"Total Power Input: {kw_input:,.2f} kW")
print(f"Estimated Number of Physical Panels: {total_panels:,} panels")
print(f"Total Weight: {total_kg:,.2f} kg")
print(f"Exact truck capacity calculated: {trucks_exact:.2f} trucks")
print(f"Total whole trucks needed: {trucks_needed}")

Enter the amount of solar panels to recycle (in kW):  1066



--- Logistics Breakdown ---
Total Power Input: 1,066.00 kW
Estimated Physical Panels: 1,777 panels
Total Weight: 53,300.00 kg
Exact truck capacity calculated: 1.84 trucks
Total whole trucks needed: 2


### Estimate Transportation Cost

In [6]:
# 